In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
sys.path.append('../../')
print(sys.path)


import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrames

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_min_reco_no_syst.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100, reprocess_df = False, reprocess_truth = False)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
#mc_bnb_evt_df = mc_bnb_evt_df[min_reco_cols_to_keep]
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

# We only need the categories for this study and the expandable systs are really big
cols_to_keep = [
    ('nu_categ', '', '', ''),
    ('genie_categ', '', '', ''),
    ('nu_categ_proton_reduced', '', '', '')
]
mc_bnb_nu_df = mc_bnb_nu_df[cols_to_keep]

keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load in time cosmic df
mc_in_time_cosmics_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_in_time_cosmics.df", keys2load, 100)
mc_in_time_cosmics_evt_df = mc_in_time_cosmics_df['cc1pi']
mc_in_time_cosmics_evt_df = mc_in_time_cosmics_evt_df[min_reco_cols_to_keep]

mc_in_time_cosmics_hdr_df = mc_in_time_cosmics_df['hdr']

'''
#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']
'''

In [ ]:

#Load CV lowE dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_lowE_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_lowE_CV.df", keys2load, 100, filter_df = False)
mc_bnb_lowE_evt_df = mc_bnb_lowE_df['cc1pi']
mc_bnb_lowE_nu_df = mc_bnb_lowE_df['nudf']
mc_bnb_lowE_hdr_df = mc_bnb_lowE_df['hdr']

mc_bnb_lowE_evt_df = mc_bnb_lowE_evt_df[min_reco_cols_to_keep]
mc_bnb_lowE_nu_df = mc_bnb_lowE_nu_df[cols_to_keep]

In [ ]:
keys2load = ["cc1pi_good", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight_quality_cut.df", keys2load, 100)
data_evt_df = data_df['cc1pi_good']
data_hdr_df = data_df['hdr']

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
'''
data_tot_pot = data_hdr_df['pot'].sum()
data_gates = data_hdr_df.nbnbinfo.sum()
'''
print("data_tot_pot: %.3e" %(data_tot_pot))
print("data tot gates : %.3e" %(data_gates))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))


#Low E
mc_bnb_lowE_tot_pot = mc_bnb_lowE_hdr_df['pot'].sum()
print("dirt_tot_pot: %.3e" %(mc_bnb_lowE_tot_pot))
mc_bnb_lowE_pot_scale = data_tot_pot / mc_bnb_lowE_tot_pot
print("dirt_pot_scale: %.3e" %(mc_bnb_lowE_pot_scale))
mc_bnb_lowE_evt_df[pot_weight_col] = mc_bnb_lowE_pot_scale * np.ones(len(mc_bnb_lowE_evt_df))




intime_gates = mc_in_time_cosmics_hdr_df[mc_in_time_cosmics_hdr_df['first_in_subrun'] == 1]['ngenevt'].sum()
print("intime cosmics data gates: {:.2e}".format(intime_gates))
f = 0.075
scale_intime_to_lightdata = (1-f)*data_gates/intime_gates
print("goal scale: {:.2f}".format(scale_intime_to_lightdata))
mc_in_time_cosmics_evt_df[pot_weight_col] = scale_intime_to_lightdata * np.ones(len(mc_in_time_cosmics_evt_df))

In [ ]:
data_evt_df = (
        data_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
        ) 

In [ ]:
mc_evt_df = mc_bnb_evt_df
if "ar23p" in bnb_path:
    print("NOT MATCHING")
    #mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_bnb_nu_df)
else:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)   
print("Finished loading")    

print("Starting concat for cosmics")    
mc_bnb_lowE_evt_df = concat_shift_first_index(mc_bnb_lowE_evt_df,mc_in_time_cosmics_evt_df)

print("TM for BNB nu df starting")
mc_bnb_lowE_evt_df = perform_truth_matching(mc_bnb_lowE_evt_df, mc_bnb_lowE_nu_df)
print("TM for BNB nu df done")

mc_bnb_lowE_evt_df = (
        mc_bnb_lowE_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
        ) 

mc_evt_df = concat_shift_first_index(mc_evt_df,mc_bnb_lowE_evt_df)

# Test background composition

In [ ]:
cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "")
cumulative_masks_data = build_event_cumulative_masks(data_evt_df, sideband = "")

#mc_cumulative_masks_sideband_pion = build_event_cumulative_masks(slc_df, sideband = "two_pions")
#mc_cumulative_masks_sideband_proton = build_event_cumulative_masks(slc_df, sideband = "proton")
#mc_cumulative_masks_sideband = mc_cumulative_masks_sideband_pion | mc_cumulative_masks_sideband_proton

In [ ]:
for name in cumulative_masks.keys():
    n_mc   = get_n_evt(mc_evt_df,   cumulative_masks[name],   use_weight=True)
    n_data = get_n_evt(data_evt_df, cumulative_masks_data[name], use_weight=False)
    print(f"{name:<15} | {n_mc:<12.2f} | {n_data:<12}")

In [ ]:
for key in ["0p", "1p", "2plusp"]:
    mask = cumulative_masks_data["energy"] & mask_dict[key](data_evt_df)
    print(f"{key}: {get_n_evt(data_evt_df, use_weight=False, mask=mask)}")

In [ ]:
slc_df = mc_evt_df

In [ ]:
#reweigth_pot = 1e20
reweigth_pot = data_tot_pot
print(reweigth_pot)
scale = reweigth_pot/data_tot_pot

In [ ]:

group_levels = ['__ntuple', 'entry', 'rec.slc..index']

obvious_cosmic_mask = mc_evt_df.slc.cut.obvious_cosmic
t0_mask = mc_evt_df.slc.cut.t0
is_inside_FV_mask =  mc_evt_df.slc.cut.inside_FV
nu_score_mask = mc_evt_df.slc.nu_score > CTE.min_nu_score
track_mask = mc_evt_df.slc.cut.track
shower_mask = mc_evt_df.slc.cut.shower 
chi2_mask = mc_evt_df.slc.cut.MIP_candidates 
angle_mask = mc_evt_df.slc.cut.angle 
proton_BDT_mask = mc_evt_df.slc.cut.proton_BDT
containment_mask = (mc_evt_df.slc.cut_var.n_exiting_pfps == 0) & (mc_evt_df.slc.cut.no_high_yz == True)
TPC_containment_mask_mask =  mc_evt_df.slc.cut.TPC_containment
michel_mask = mc_evt_df.slc.cut.michel 
extra_pion_mask = mc_evt_df.slc.cut.extra_pion 
energy_mask = mc_evt_df.slc.cut.energy

# 1. Define the order of cuts
cut_sequence = [
    ("cosmic", obvious_cosmic_mask),
    ("cosmic_rejection", nu_score_mask  & t0_mask & is_inside_FV_mask),
    ("track", track_mask),
    ("chi2", chi2_mask),
    ("shower", shower_mask),
    ("angle", angle_mask),
    ("proton_BDT", proton_BDT_mask),
    ("containment", containment_mask),
    ("TPC_containment", TPC_containment_mask_mask ),
    ("michel", michel_mask),
    ("extra_pion", extra_pion_mask),
    ("energy", energy_mask)
]

# 2. Build the cumulative masks
cumulative_masks_plot = {}
current_mask = None

for name, mask in cut_sequence:
    if current_mask is None:
        current_mask = mask
    else:
        current_mask = current_mask & mask
    cumulative_masks_plot[name] = current_mask


plot_data = []
for stage_name, mask in cumulative_masks_plot.items():


    # 1. Collapse the 4-level mask into a 3-level mask
    collapsed_mask = mask.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()

    # 2. Align the collapsed mask with slc_df
    final_mask = collapsed_mask.reindex(slc_df.index, fill_value=False)

    # 3. Filter and Aggregate
    filtered_categs = slc_df.loc[final_mask, [('truth','nu_categ','','','',''), pot_weight_col]]
    counts = filtered_categs.groupby([('truth','nu_categ','','','','')])[[pot_weight_col]].sum()*scale

    counts = counts.iloc[:, 0]
    counts.name = stage_name
    plot_data.append(counts)

# Create the initial DataFrame
df_results_plot = pd.concat(plot_data, axis=1).T.fillna(0)

# Ensure CC1pi is the first column before renaming
signal_tag = 'CC1pi'
if signal_tag in df_results_plot.columns:
    cols = [signal_tag] + [c for c in df_results_plot.columns if c != signal_tag]
    df_results_plot = df_results_plot[cols]

df_results_plot_renamed = df_results_plot.rename(columns=bkg_name_nice_map).rename(index=cut_name_nice_map)

In [ ]:
cut_name_nice_map = {
    "cosmic_rejection": "Cosmic rejection",
    "cosmic": "Initial sample",
    "t0": "Timing",
    "FV": "FV",
    "nu_score": "Nu Score Cut",
    "track": "2 Tracks",
    "shower": "No shower",
    "chi2": "2 MIP candidates",
    "angle": "Angle constraint",
    "proton_BDT": "HE proton removal",
    "containment": "No particles exiting or in high-yz",
    "michel": "Michel removal",
    "extra_pion": "Extra pion removal",
    "energy": "Kinematic constraints",
    "MIP_refinement": "Michel & extra pion removal",
    "TPC_containment": "Local TPC Containment",
}

In [ ]:
# --- CONFIGURATION TOGGLES ---
USE_LOG_SCALE = True
X_MIN_LOG = 100
# ----------------------------

# 2. MATCH COLORS TO CURRENT COLUMNS
plot_colors = [category_colors.get(col, "#000000") for col in df_results_plot.columns]

# --- 1. Prepare Data for Plotting ---
df_results_plot_renamed = df_results_plot.rename(columns=bkg_name_nice_map).rename(index=cut_name_nice_map)

# Create the side-by-side figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9), sharey=True)

# --- 2. Efficiency Plot (Left) ---
df_plot_efficiency = df_results_plot_renamed[df_results_plot_renamed.sum(axis=1) > 0]

df_plot_efficiency.plot(
    kind='barh',
    stacked=True,
    ax=ax1,
    color=plot_colors,
    legend=True 
)

if USE_LOG_SCALE:
    ax1.set_xscale('log')
    max_val = df_plot_efficiency.sum(axis=1).max() * 5 
    ax1.set_xlim(X_MIN_LOG, max_val)

ax1.set_xlabel("Candidate Events", fontsize=18)
ax1.set_ylabel("") 
ax1.grid(True, which="both", ls="-", alpha=0.2)
ax1.tick_params(axis='both', labelsize=16)

# --- MOVE LEGEND TO BOTTOM RIGHT ---
leg = ax1.legend(
    loc='lower right',
    bbox_to_anchor=(0.98, 0.02),
    fontsize=16, 
    framealpha=1.0, 
    edgecolor='black', 
    fancybox=False
)

# --- 3. Purity Plot (Right) ---
df_purity = df_results_plot_renamed.div(df_results_plot_renamed.sum(axis=1), axis=0)

df_purity.plot(
    kind='barh',
    stacked=True,
    ax=ax2,
    color=plot_colors,
    legend=False 
)

ax2.set_xlabel("Fraction of Total Events", fontsize=18)
ax2.set_ylabel("")
ax2.tick_params(axis='both', labelsize=16)

# --- INVERT THE ORDER OF THE BARS ---
ax2.invert_yaxis()

ax2.set_xlim(0, 1)
ticks = np.arange(0, 1.1, 0.1)
ax2.set_xticks(ticks)
ax2.xaxis.set_minor_locator(MultipleLocator(0.05))

ax2.grid(True, which='major', axis='x', ls='--', alpha=0.8, color='gray', zorder=0)
ax2.grid(True, which='minor', axis='x', ls='--', alpha=0.4, color='gray', zorder=0)

# Final Layout Adjustments
plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.subplots_adjust(wspace=0.08) 

# --- FIXED SECTION: DYNAMICALLY PLACING LABELS OVER THE LEGEND ---
# 1. Force the canvas engine to evaluate geometry transformations *after* tight_layout/adjust
fig.canvas.draw()
renderer = fig.canvas.get_renderer()

# 2. Extract bounding box details
leg_bbox = leg.get_window_extent(renderer)
inv_axes = ax1.transAxes.inverted()

# Get the exact top right corner coordinate of the legend box inside ax1
top_right_in_axes = inv_axes.transform((leg_bbox.x1, leg_bbox.y1))
leg_top_x = top_right_in_axes[0]
leg_top_y = top_right_in_axes[1]

# 3. Stack upwards cleanly using va='bottom' so text cannot overlap the box layout.
# Ensure your underlying functions use `va='bottom'` or don't override it to `top`.
plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.subplots_adjust(wspace=0.08)
plt.draw()

# Convert legend bbox to axes fraction — DPI-independent
renderer  = fig.canvas.get_renderer()
leg_bbox  = leg.get_window_extent(renderer)
inv_axes  = ax1.transAxes.inverted()

leg_top_x, leg_top_y = inv_axes.transform((leg_bbox.x1, leg_bbox.y1))

# Convert fixed offsets to axes fraction using actual axes height in pixels
axes_height_px = ax1.get_window_extent(renderer).height
offset_per_unit = 1.0 / axes_height_px

# 14pt font at 1.5x line spacing
line_height_ax = (14 * 1.5) * offset_per_unit

for i, (fn, fs) in enumerate([
    (add_exposure_text,      15.5),
    (add_genie_version_text, 15.5),
    (add_approval_text,      18  ),
]):
    kwargs = dict(
        textloc_x=leg_top_x,
        textloc_y=leg_top_y + (i + 1) * line_height_ax * 1.2,
        textloc_ha='right',
        ax=ax1,
        fontsize=fs,
    )
    if fn == add_exposure_text:
        kwargs['data_pot'] = reweigth_pot
    if fn == add_approval_text:
        kwargs['approval'] = "AnalysisInProgress"
    fn(**kwargs)

f_name = f"cc1pi_selection_summary.pdf" 

# --- SAVE HANDLING (MUST BE BEFORE plt.show()) ---
parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/PurEffGraphs"
if not os.path.exists(parent_path):
    os.makedirs(parent_path)
    
save_full_path = os.path.join(parent_path, f_name)




# Save BEFORE plt.show() — show() resets the canvas
fig.savefig(save_full_path, format='pdf', bbox_inches='tight')
plt.show()



# Finally, display to screen
plt.show()

In [ ]:
plot_data = []
for stage_name, mask in cumulative_masks.items():

    # 1. Collapse the 4-level mask into a 3-level mask
    collapsed_mask = mask.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()

    # 2. Align the collapsed mask with slc_df
    final_mask = collapsed_mask.reindex(slc_df.index, fill_value=False)

    # 3. Filter and Aggregate
    filtered_categs = slc_df.loc[final_mask, [('truth','nu_categ','','','',''), pot_weight_col]]
    counts = filtered_categs.groupby([('truth','nu_categ','','','','')])[[pot_weight_col]].sum()*scale

    counts = counts.iloc[:, 0]
    counts.name = stage_name
    plot_data.append(counts)

# Create the initial DataFrame
df_results = pd.concat(plot_data, axis=1).T.fillna(0)

# Ensure CC1pi is the first column before renaming
signal_tag = 'CC1pi'
if signal_tag in df_results.columns:
    cols = [signal_tag] + [c for c in df_results.columns if c != signal_tag]
    df_results = df_results[cols]


df_results_renamed = df_results.rename(columns=bkg_name_nice_map).rename(index=cut_name_nice_map)

In [ ]:

print(scale)

# Assuming 'CC1Pi' is your signal column
signal_col = '$\\nu_{\\mu}$CC1$\\pi^{\\pm}$'
#signal_col = "$\\nu_{\\mu}$CC0$\\pi^{\\pm}$1p"

# 1. Calculate Total Events per stage
total_per_stage = df_results_renamed.sum(axis=1)

# 2. Calculate Purity: Signal / Total at each stage
purity = (df_results_renamed[signal_col] / total_per_stage) * 100

# 3. Calculate Efficiency: Signal at stage / Signal at first stage
# (Recreating num_events_0 logic)
initial_signal_count = df_results_renamed[signal_col].iloc[0]
efficiency = (df_results_renamed[signal_col] / initial_signal_count) * 100

# 4. Create a summary table for easy viewing
summary_df = pd.DataFrame({
    'Signal_Events': df_results_renamed[signal_col],
    'Total_Events': total_per_stage,
    'Purity (%)': purity,
    'Efficiency (%)': efficiency
})

print(summary_df)

In [ ]:
import numpy as np

print(f"{'Cut Name':<20} | {'Purity':<10} | {'Efficiency':<10} | {'S/sqrt(S + B)':<10} | {'pur * eff':<10}")
print("-" * 75)

for stage in df_results_renamed.index:
    n_signal = df_results_renamed.loc[stage, signal_col]*scale
    n_total = total_per_stage.loc[stage]*scale
    n_bkg = n_total - n_signal
    
    pur = (n_signal / n_total) * 100 if n_total > 0 else 0
    eff = (n_signal / initial_signal_count) * 100
    pur_eff = pur*eff/100
    s_sqrt_b = n_signal / np.sqrt(n_bkg + n_signal) if n_bkg > 0 else np.nan
    
    print(f"{stage:<20} | {pur:>8.2f}% | {eff:>8.2f}% | {s_sqrt_b:>8.2f} | {pur_eff:8.2f}") 

In [ ]:
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"].groupby(['__ntuple', 'entry', 'rec.slc..index']).first()], ('truth','nu_categ','','','',''))

In [ ]:
print("0p")
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"].groupby(['__ntuple', 'entry', 'rec.slc..index']).first() & mask_dict["0p"](slc_df)], ('truth','nu_categ_proton_reduced','','','',''))
print("1p")
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"].groupby(['__ntuple', 'entry', 'rec.slc..index']).first() & mask_dict["1p"](slc_df)], ('truth','nu_categ_proton_reduced','','','',''))
print("2p+")
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"].groupby(['__ntuple', 'entry', 'rec.slc..index']).first() & mask_dict["2plusp"](slc_df)], ('truth','nu_categ_proton_reduced','','','',''))